# Tutorial 2: Market Realism — How the Stochastic Layers Work

Spectral-Env-Core doesn't just add noise to a trend line. It combines **six independent stochastic layers**, each modelling a distinct market phenomenon. This tutorial visualises each layer in isolation and then shows the combined result.

## Why layered stochastic models matter

Real asset prices exhibit several properties simultaneously:

| Property | Plain GBM? | Spectral-Env-Core |
|---|---|---|
| Fat tails (extreme events) | ❌ (Gaussian) | ✅ Student-t |
| Volatility clustering | ❌ (constant σ) | ✅ GARCH(1,1) |
| Serial correlation | ❌ (i.i.d.) | ✅ AR(1) |
| Regime changes | ❌ | ✅ Markov switching |
| Sudden crashes/spikes | ❌ | ✅ Jump diffusion |
| Cross-asset dependence | ❌ | ✅ Cholesky correlation |

An RL agent trained only on GBM learns nothing useful — the optimal policy on a random walk is "don't trade" (fees eat any edge). The layered model creates exploitable structure: momentum, mean-reversion, regime awareness, and jump timing all become learnable strategies.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from spectral_env_core import SpectralTradingEnv

## Layer 1: Student-t Fat Tails

Real markets have "tail events" — large moves that occur far more often than a Gaussian distribution predicts. The 2008 crash, the 2020 COVID drop, flash crashes — all are essentially impossible under normal distributions.

Spectral-Env-Core uses Student-t distributed innovations with configurable degrees of freedom (`df`). Lower `df` = fatter tails.

In [ ]:
# Compare Gaussian vs Student-t(df=10) tails
np.random.seed(42)
n = 50_000
gaussian = np.random.standard_normal(n)
student_t = np.random.standard_t(df=10, size=n) / np.sqrt(10 / 8)  # variance-corrected

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(gaussian, bins=100, alpha=0.6, density=True, label='Gaussian', color='blue')
axes[0].hist(student_t, bins=100, alpha=0.6, density=True, label='Student-t (df=10)', color='red')
axes[0].set_xlim(-6, 6)
axes[0].set_title('Return Distributions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# QQ plot
sorted_t = np.sort(student_t)
theoretical = np.sort(gaussian)
axes[1].scatter(theoretical[::50], sorted_t[::50], s=5, alpha=0.5)
axes[1].plot([-5, 5], [-5, 5], 'r--', lw=1)
axes[1].set_title('QQ Plot: Student-t vs Gaussian')
axes[1].set_xlabel('Gaussian quantiles')
axes[1].set_ylabel('Student-t quantiles')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Count extreme events
threshold = 3.0
gauss_extremes = (np.abs(gaussian) > threshold).sum()
t_extremes = (np.abs(student_t) > threshold).sum()
print(f"Events beyond ±3σ:")
print(f"  Gaussian:    {gauss_extremes} ({gauss_extremes/n*100:.2f}%)")
print(f"  Student-t:   {t_extremes} ({t_extremes/n*100:.2f}%)")
print(f"  Ratio:       {t_extremes/gauss_extremes:.1f}× more extreme events with Student-t")

## Layer 2: GARCH(1,1) — Volatility Clustering

In real markets, large moves tend to follow large moves (high vol persists) and small moves follow small moves (low vol persists). This is called *volatility clustering* and it's one of the most robust empirical facts in finance.

GARCH(1,1) models this via a time-varying variance: `h_t = ω + α·z²_{t-1} + β·h_{t-1}`

- `α` (garch_alpha): how much yesterday's shock affects today's volatility
- `β` (garch_beta): how persistent volatility is once elevated
- `α + β` close to 1.0: volatility takes a long time to revert to baseline

In [ ]:
# Generate paths with and without GARCH
env_no_garch = SpectralTradingEnv(
    num_steps=500, volatility=0.25, garch_alpha=0.0, garch_beta=0.0,
    jump_intensity=0.0, randomize_phi=False, phi=0.0,
    regime_switch_prob=0.0,
)
env_garch = SpectralTradingEnv(
    num_steps=500, volatility=0.25, garch_alpha=0.10, garch_beta=0.85,
    jump_intensity=0.0, randomize_phi=False, phi=0.0,
    regime_switch_prob=0.0,
)

env_no_garch.reset(seed=7)
env_garch.reset(seed=7)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Compute log returns from paths
path_no_garch = env_no_garch.brownian_path[:, 0]
path_garch    = env_garch.brownian_path[:, 0]
ret_no_garch  = np.diff(np.log(path_no_garch))
ret_garch     = np.diff(np.log(path_garch))

axes[0].bar(range(len(ret_no_garch)), ret_no_garch, color='steelblue', width=1.0)
axes[0].set_title('Returns WITHOUT GARCH (constant volatility)', fontsize=11)
axes[0].set_ylabel('Log Return')
axes[0].set_ylim(-0.08, 0.08)
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(ret_garch)), ret_garch, color='crimson', width=1.0)
axes[1].set_title('Returns WITH GARCH (α=0.10, β=0.85) — notice the clustering', fontsize=11)
axes[1].set_ylabel('Log Return')
axes[1].set_xlabel('Step')
axes[1].set_ylim(-0.08, 0.08)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The GARCH path shows clear periods of high and low volatility —")
print("exactly what real equity returns look like.")

## Layer 3: Markov Regime Switching

Markets don't just have one "mode" — they switch between distinct states (bull, bear, sideways). Each regime has its own drift and volatility multiplier.

The environment uses a Markov chain with configurable transition probability. This creates realistic multi-month trend persistence that an RL agent can learn to detect and exploit.

In [ ]:
env_regime = SpectralTradingEnv(
    num_steps=500,
    initial_price=100.0,
    volatility=0.20,
    drift=0.15,
    garch_alpha=0.05, garch_beta=0.85,
    regime_drift_mults=(1.5, -0.8),   # bull: strong up, bear: moderate down
    regime_vol_mults=(0.6, 2.0),      # bull: low vol, bear: high vol
    regime_switch_prob=0.03,           # ~15 switches per 500 steps
    jump_intensity=0.0,
    randomize_phi=False, phi=0.0,
)

env_regime.reset(seed=12)
path = env_regime.brownian_path[:, 0]
regimes = env_regime.regime_sequence

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

axes[0].plot(path, color='black', lw=1.5)
# Shade regimes
for t in range(len(regimes)):
    color = 'lightgreen' if regimes[t] == 0 else 'lightsalmon'
    axes[0].axvspan(t, t+1, alpha=0.3, color=color, lw=0)
axes[0].set_title('Price Path with Regime Overlay (green=bull, red=bear)', fontsize=12)
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

axes[1].step(range(len(regimes)), regimes, color='steelblue', lw=1.5)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Bull (R0)', 'Bear (R1)'])
axes[1].set_xlabel('Step')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Regime 0 (bull): drift ×1.5, vol ×0.6 — trending up, low noise")
print(f"Regime 1 (bear): drift ×-0.8, vol ×2.0 — trending down, high noise")
print(f"\nAn RL agent that learns to detect regime shifts and adjust position size")
print(f"accordingly will significantly outperform one that treats all steps identically.")

## Layer 4: Jump Diffusion (Merton 1976)

Even with fat tails and GARCH, continuous models can't produce the *discontinuous* price gaps seen in real markets — earnings surprises, flash crashes, overnight gaps. Jump diffusion adds discrete Poisson-distributed events with configurable frequency and severity.

This is a **Pro-only feature** — it's what separates agents that are merely profitable from agents that are *robust*.

In [ ]:
env_jumps = SpectralTradingEnv(
    num_steps=500,
    initial_price=100.0,
    volatility=0.20,
    drift=0.10,
    garch_alpha=0.05, garch_beta=0.85,
    jump_intensity=3.0,     # ~3 jumps per year
    jump_mean=-0.04,        # average jump is -4% (crash-biased)
    jump_std=0.06,          # some jumps are positive (earnings beats)
    randomize_phi=False, phi=0.0,
)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

for i in range(10):
    env_jumps.reset(seed=i)
    axes[0].plot(env_jumps.brownian_path[:, 0], alpha=0.6, lw=1)

axes[0].set_title('10 Episodes WITH Jump Diffusion (λ=3, μ=-4%)', fontsize=12)
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# Without jumps for comparison
env_no_jumps = SpectralTradingEnv(
    num_steps=500, initial_price=100.0, volatility=0.20, drift=0.10,
    garch_alpha=0.05, garch_beta=0.85, jump_intensity=0.0,
    randomize_phi=False, phi=0.0,
)
for i in range(10):
    env_no_jumps.reset(seed=i)
    axes[1].plot(env_no_jumps.brownian_path[:, 0], alpha=0.6, lw=1)

axes[1].set_title('Same 10 Seeds WITHOUT Jump Diffusion', fontsize=12)
axes[1].set_ylabel('Price ($)')
axes[1].set_xlabel('Step')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The jump paths show sudden discontinuities — exactly the events that")
print("destroy agents trained only on smooth price processes.")

## Design Philosophy: Why Each Layer Exists

Each layer wasn't added for mathematical elegance — it was added because **RL agents exploit specific patterns** and we need the environment to produce those patterns realistically:

| Layer | Pattern it creates | Strategy an agent can learn |
|---|---|---|
| Student-t | Extreme returns are more common | Risk management, position sizing for tail events |
| GARCH | Volatility persists | Reduce position during high-vol, increase during low-vol |
| AR(1) | Momentum / mean-reversion | Trend-following or contrarian timing |
| Regime switching | Bull/bear persistence | Regime detection → adapt strategy |
| Jump diffusion | Sudden crashes | Stop-loss timing, crash hedging |
| Correlation | Co-movement across assets | Portfolio diversification, pair trading |

An environment without these layers produces a random walk. The optimal policy on a random walk is trivial: hold cash. That's why most toy financial RL environments produce agents that learn nothing.

---

**Next tutorial:** [03 — Multi-Asset Portfolios](03_multi_asset_portfolios.ipynb) — train an agent to allocate across correlated assets.